# Self-Attention and the Transformer Block

> In the previous section, we added positional information to each token's vector. This completes the work of the input layer: each token now carries both semantic information ("what this word is") and positional information ("which position this word sits at"). But there is still a problem—these vectors are computed independently, with no communication between them. In other words, the model at this point does not know how tokens relate to each other.
>
> This section starts from that problem and introduces the Attention mechanism, letting each token extract information from context on demand. We will first break down the Attention computation by hand, then progressively add causal masking, the multi-head mechanism, the feed-forward network, residual connections, and LayerNorm, and finally assemble everything into a Transformer Block.

Take "the cat sat on the mat" as an example. Without looking at context, "sat" is just a vector representing "sit". But with context, the model can know who is sitting (cat) and where it is sitting (mat). Consider "bank" again: in "river bank" it means the riverside, in "bank account" it means a financial institution—the same four letters, with meaning determined by the surrounding words.

These examples point to a core requirement: tokens need to exchange information. The overall Transformer architecture is therefore divided into three stages:

```
Input layer:   Tokenizer -> Embedding -> Position Encoding    <- completed in the previous three sections
Core layer:    N Transformer Blocks                              <- focus of this section
Output layer:  Linear -> logits                                   <- next section
```

What we will build in this section is the middle Transformer Block. The Block's core mechanism is called Attention—each token first judges which parts of the context are more relevant, then mixes information weighted by that relevance. Specifically, we will build it step by step in the following order:

1. **Scaled Dot-Product Attention**: compute attention weights from three vectors Q, K, V and then perform a weighted mix—this is the core computation
2. **Causal Mask**: mask future positions so that GPT can only see preceding tokens
3. **Multi-Head Attention**: multiple sets of Q/K/V working in parallel, extracting context information from different angles
4. **Transformer Block**: assemble Multi-Head Attention and the feed-forward network (FFN) together, with residual connections and LayerNorm

Each step adds only one new component on top of the previous one, and the final step assembles a complete Block.

## 1. Intuition Behind Attention

The output of Attention can be described by a set of weights. Suppose the model is processing "the cat sat on the mat". When it processes "sat", the attention weights it produces might be:

```
      the  cat  sat  on  the  mat
sat: 0.05 0.35 0.10 0.05 0.05 0.40
```

These numbers sum to 1. "cat" and "mat" have large weights, meaning "sat" reads more information from them. The resulting new vector for "sat" then blends in the context of "a cat sitting on a mat".

Computing these attention proportions is the core problem Attention solves. It introduces three vectors—Q (Query), K (Key), and V (Value)—playing three roles: "asking a question", "providing a label", and "offering content":

- Q (Query): What am I looking for? For instance, "sat" might be asking: "Who is performing this action? Where is the action taking place?"
- K (Key): What label do I carry? For instance, "cat"'s Key might signal to others: "I am the doer of an action."
- V (Value): What content can I provide? If "cat" is attended to, what actually gets mixed into the output is its Value.

A simple analogy is looking up references: Query is the question you want to answer, Key is the title or tag of each document, and Value is the body of the document. You first use Query and Key to determine which documents are relevant, then read in the Value of the relevant documents in proportion.

In Self-Attention, Q, K, and V are all computed from the same input matrix X, just multiplied by three different weight matrices:

```
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

The same input goes through three different linear projections and takes on three different roles. Below we construct X in code and then compute Attention step by step.

In [ ]:
# === Vocabulary → sentence → Token IDs → Embedding lookup → X ===
import torch
import torch.nn as nn
_ = torch.manual_seed(42)
vocab = {"the": 0, "a": 1, "cat": 2, "dog": 3, "sat": 4,
         "ran": 5, "on": 6, "mat": 7, "[PAD]": 8, "[UNK]": 9}
vocab_size = len(vocab)
id2word = {v: k for k, v in vocab.items()}

# Sentence → Token IDs
sentence = ["the", "cat", "sat"]
token_ids = [vocab[w] for w in sentence]  # [0, 2, 4]

# Token IDs → Embedding lookup → X
d_model = 4
embedding = nn.Embedding(vocab_size, d_model)
token_ids_tensor = torch.tensor(token_ids)  # [3]
X = embedding(token_ids_tensor)             # [3, 4]

print(f"Vocabulary size: {vocab_size}, sentence: {' '.join(sentence)}")
print(f"Token IDs: {token_ids}")
print(f"X shape: {list(X.shape)} ← [seq_len=3, d_model=4]")
print(f"\nX (from an Embedding lookup, not randn):\n{X}")
print(f"\nExplanation: X[0]='the' → {X[0].tolist()}")
print(f"             X[1]='cat' → {X[1].tolist()}")
print(f"             X[2]='sat' → {X[2].tolist()}")


In [ ]:
# === Start from X and compute Q, K, and V ===
import torch.nn as nn
seq_len = X.shape[0]  # = 3
d_k = 4

# Q/K/V share the same input X but use different projection matrices
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)  # [3, 4] — each Token's query
K = W_K(X)  # [3, 4] — each Token's key
V = W_V(X)  # [3, 4] — each Token's content

print(f"X shape: {X.shape} → Q/K/V shape: {Q.shape}")
print("→ Q, K, and V all come from X through different matrices")


## 2. Scaled Dot-Product Attention

Attention computation has four steps. The first step uses Q and K to compute relevance scores—row i, column j represents how interested token i is in token j. A larger dot product means a stronger match.

In [ ]:
# Step 1: Attention scores = Q × K^T
# Entry (i, j) is Token i's raw relevance score for Token j
attention_scores = Q @ K.T  # [3, 4] @ [4, 3] = [3, 3]

print(f"Attention score matrix {list(attention_scores.shape)} = [{seq_len}×{seq_len}]:")
print(attention_scores)
print(f"\nRow i contains Token {list(range(seq_len))}'s scores for every Token")


**Scaling**

Dot products can be large. When values are too large, softmax becomes overconfident and training becomes unstable.

So we divide by `\u221ad_k` to keep the scores more stable.

Why \u221ad_k and not some other number? This can be understood from the perspective of variance. Suppose each element of Q and K is an independent random variable with mean 0 and variance 1. Then $q_i k_i$ also has variance 1, and the dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$.

In other words, the higher the dimension, the larger the typical absolute value of the dot product. When $d_k = 64$, the typical magnitude of the dot product is around 8 ($\sqrt{64} = 8$), and softmax inputs near \u00b18 are already close to saturation, where gradients become very small.

Dividing by $\sqrt{d_k}$ effectively pulls the variance of the dot product back to 1, keeping softmax in a region with sufficient gradients.

In [ ]:
# Step 2: divide by sqrt(d_k) so large dot products do not saturate Softmax
import math
d_k = Q.shape[-1]
scaled_scores = attention_scores / math.sqrt(d_k)

print(f"Scale factor: sqrt({d_k}) = {math.sqrt(d_k):.2f}")
print(f"Token 0 before scaling: {attention_scores[0].tolist()}")
print(f"Token 0 after scaling:  {scaled_scores[0].tolist()} ← smaller values, same ordering")


The third step applies softmax to turn each row of scores into probabilities—each row sums to 1, representing how much attention this token pays to each other token.

The fourth step uses these weights to mix V. Whoever has a larger weight contributes more information. This way, each token's output blends in the context it attended to.

In [ ]:
# Step 3: Softmax converts scores to probabilities whose rows sum to 1
import torch.nn.functional as F
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Attention weight matrix {list(attention_weights.shape)}:")
print(attention_weights)

# Verify that every row sums to 1
print(f"\nRow sums: {attention_weights.sum(dim=-1).tolist()} ← all are 1.0")


In [ ]:
# Step 4: weighted sum—use Attention weights to mix V
output = attention_weights @ V  # [3, 3] @ [3, 4] = [3, 4]

print(f"Output shape: {list(output.shape)} = [{seq_len}, {d_model}]")
print(f"\nInput Token 0:  {X[0].tolist()}")
print(f"Output Token 0: {output[0].tolist()}")
print("→ They differ because Token 0 mixes information from Tokens 1 and 2")


### Four-Step Summary

**Step 1 — Linear projections generate Q/K/V:**

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

The same input $X$ multiplied by three sets of weights produces three roles: "asking a question", "label", and "content".

**Step 2 — Compute relevance scores:**

$$S = QK^\top$$

Row $i$, column $j$ is the raw match score of token $i$ toward token $j$. A larger dot product means a stronger relevance.

**Step 3 — Scale + Softmax:**

$$A = \text{softmax}\!\left(\frac{S}{\sqrt{d_k}}\right)$$

Dividing by $\sqrt{d_k}$ prevents the dot product from growing too large and saturating softmax. After softmax, each row sums to 1, producing a set of attention weights.

**Step 4 — Weighted sum:**

$$\text{output} = AV$$

Whoever has a larger weight contributes more information. Each token's output vector blends in the context it attended to.

Combined into a single formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

## 3. Causal Masking

The Scaled Dot-Product Attention we implemented in the previous section lets each token see every token in the sequence, including those after it. This is fine when understanding the meaning of a whole sentence. But GPT is a generative model—it works by **writing one word at a time, moving forward**. Training follows the same logic: given a sentence, it practices "seeing the preceding words and guessing the next word".

### Why GPT Practices "Guessing the Next Word"

Let's step back: what does GPT ultimately do? Given an opening, it continues writing forward. For example, given the input "Once upon a time, there was a mountain, and in the mountain there was a", it should continue with "temple". In other words, GPT's entire capability boils down to one thing: **given the text written so far, judge what the next word most likely is**.

If it can always guess the next word correctly, it can keep writing and produce coherent long texts. So how does it develop this ability? It's like learning to write essays: the teacher gives you the start of a sentence, "Spring has come, the ice and snow", and asks you to fill in the next word. You write "melt"—correct; you write "thaw"—also correct; you write "keyboard"—not so right. Through many such exercises, you gradually learn what kind of word should follow what kind of preceding context.

GPT's training works exactly the same way. Given a sentence "the cat sat on the mat", it repeatedly practices:

```
seeing "the"                -> guess the next word (answer is cat)
seeing "the cat"            -> guess the next word (answer is sat)
seeing "the cat sat"        -> guess the next word (answer is on)
seeing "the cat sat on"     -> guess the next word (answer is the)
seeing "the cat sat on the" -> guess the next word (answer is mat)
```

A single sentence yields 5 practice examples. The training data contains billions of sentences, each broken up into many "read the preceding context, guess the next word" exercises. With enough practice, the model learns: "the" is usually followed by a noun, "cat sat" is usually followed by a preposition, "on" is usually followed by "the", and so on.

### The Problem: Attention Lets Every Token See Every Other Token

These guessing tasks are completed **simultaneously**. The model feeds in 6 tokens together, and one forward pass computes 6 output vectors, each used to guess the next word at its own position.

Recall the computation of Scaled Dot-Product Attention. The first step computes the score matrix $S = QK^\top$. For 3 tokens (the, cat, sat), this matrix is $3 \times 3$:

```
         the   cat   sat
the:    [0.2,  0.3,  0.1]
cat:    [0.3,  0.5,  0.8]
sat:    [0.1,  0.2,  0.6]
```

Each row represents one token's attention toward the others. Row 1 is cat's attention: cat's score toward the is 0.3, toward itself is 0.5, toward sat is 0.8. After softmax, these scores become weights (summing to 1), and cat's output vector is the weighted mix of the Value vectors of the, cat, and sat.

But the training objective is for cat's output vector to predict the next word—and the next word happens to be sat. sat occupies the largest share of cat's attention weights, meaning cat's output mixes in a large amount of sat's information. When the model uses this output to predict "what is the next word", it is of course very easy to guess sat. But the model has not actually learned to infer cat from the—it has simply seen the answer directly.

An analogy: during an exam, the question is "the \_\_\_ sat", and the answer is cat. If you are allowed to peek at the next word, then as soon as you see sat you know the blank is most likely the subject cat—the score looks good, but give this student a question where they cannot peek at the answer and they will not know how to solve it.

### Masking: Let Each Token See Only Itself and Preceding Tokens

The solution is straightforward: since the problem is that tokens see the answer ahead, just mask out all the scores after them.

token 0 (the) should only see the, because the next word it has to guess is at position 1, and it must not see information at positions 1 and 2.

token 1 (cat) should only see the and cat, because the next word it has to guess is at position 2, and it must not see information at position 2.

token 2 (sat) can see everything, because the next word it has to guess is at position 3, and positions 0-2 are all "before it", with nothing leaked.

Drawing "who can see whom" as a matrix, where 1 means "allowed to see" and 0 means "masked":

```
         the  cat  sat
the:    [ 1,   0,   0 ]    <- row 0: only position 0 is 1
cat:    [ 1,   1,   0 ]    <- row 1: positions 0 and 1 are 1
sat:    [ 1,   1,   1 ]    <- row 2: all are 1
```

This matrix happens to be a lower-triangular matrix—everything above the diagonal is 0.

### How Masking Acts on the Scores

Applying this mask to the raw score matrix, masked positions are replaced with $-\infty$:

```
raw scores:               masked:
[0.2, 0.3, 0.1]  ->  [0.2,  -inf,  -inf]   <- the's row: mask cat and sat
[0.3, 0.5, 0.8]  ->  [0.3,  0.5,  -inf]    <- cat's row: mask sat
[0.1, 0.2, 0.6]  ->  [0.1,  0.2,  0.6]     <- sat's row: no masking
```

Then softmax is applied to each row. The property of softmax is: positions with input $-\infty$ produce output 0. Masked positions vanish completely from the weights:

```
masked:                    softmax weights:
[0.2,  -inf,  -inf]  ->  [1.00, 0.00, 0.00]   <- the 100% uses only itself
[0.3,  0.5,  -inf]   ->  [0.45, 0.55, 0.00]   <- cat uses the(45%) and itself(55%), sat weight=0
[0.1,  0.2,  0.6]    ->  [0.22, 0.27, 0.51]   <- sat sees all
```

In cat's row, the weight on sat becomes 0. cat's output vector contains only information from the and cat itself, with nothing of sat. When the model uses this output to predict the next word, it must genuinely learn to infer cat from the, and can no longer rely on peeking.

Implementing this mask in code requires only generating a lower-triangular matrix.

In [ ]:
# Apply causal masking and Softmax to the scaled scores from Section 2
import torch
import torch.nn.functional as F

print("=== Step 1: Scaled scores before masking ===")
print(f"scaled_scores:\n{scaled_scores.detach()}\n")

# Create a lower-triangular causal mask
seq_len = scaled_scores.shape[0]
mask = torch.tril(torch.ones(seq_len, seq_len))
print(f"=== Step 2: Causal Mask ({seq_len}×{seq_len}) ===")
print(f"mask (1=visible, 0=hidden):\n{mask.int()}\n")

# Replace every masked position with negative infinity
masked_scores = scaled_scores.masked_fill(mask == 0, float('-inf'))
print("=== Step 3: Scores after masking ===")
print(f"Replace zero-mask positions with -inf:\n{masked_scores.detach()}\n")

masked_weights = F.softmax(masked_scores, dim=-1)
print("=== Step 4: Softmax after masking ===")
print(f"Masked weights:\n{masked_weights.detach()}\n")

# Compare weights before and after masking
unmasked_weights = F.softmax(scaled_scores, dim=-1)
print("=== Comparison: before and after masking ===")
print(f"{'':>8} {'unmasked':>30}  {'masked':>30}")
labels = ['the', 'cat', 'sat']
for i in range(seq_len):
    row_before = [f"{v:.3f}" for v in unmasked_weights[i].tolist()]
    row_after = [f"{v:.3f}" for v in masked_weights[i].tolist()]
    print(f"{labels[i]:>8} {str(row_before):>30}  {str(row_after):>30}")

print()
print("Key observation:")
print(f"  Without a mask, cat attends to sat with weight {unmasked_weights[1,2]:.3f}")
print(f"  With a mask, cat attends to sat with weight {masked_weights[1,2]:.3f}")
print("  The cat position can attend only to 'the' and itself")


## 4. Multi-Head Attention

### What Is a "Head"

Review the code from section 2. We defined three linear transforms `W_Q`, `W_K`, `W_V` (all `nn.Linear(d_model, d_k)`), used them to project the input $X$ into Q/K/V, computed Attention once, and got a $3 \times 3$ weight matrix:

```python
W_Q = nn.Linear(d_model, d_k, bias=False)   # code from section 2
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)
Q = W_Q(X)   # [3, 4]
K = W_K(X)   # [3, 4]
V = W_V(X)   # [3, 4]
# -> scores -> softmax -> weights -> output
```

Two dimensions appear here. `d_model` is the unified vector dimension inside the model—the Embedding output is `d_model`-dimensional, the Transformer Block's input and output are also `d_model`-dimensional, and the whole model keeps this width from start to finish. Different frameworks give this dimension different names:

| Name | Source | Example |
|:---|:---|:---|
| `d_model` | original Transformer paper, PyTorch `nn.Transformer` | $d_{model} = 512$ in the paper |
| `embed_dim` | PyTorch `nn.MultiheadAttention` | `embed_dim=768` |
| `hidden_size` | HuggingFace Transformers (BERT, LLaMA, etc.) | `hidden_size=4096` |
| `n_embd` | OpenAI GPT-series config files | `n_embd=768` |

All four names refer to the same thing: the dimension of the token vector. `d_k` is the dimension after the Q/K/V projection. In section 2 we had `d_model = d_k = 4`, so the two are equal. But in multi-head attention they will differ, as we will see below.

This group of `W_Q, W_K, W_V` plus the subsequent Attention computation is one "head". In code, one head is three lines of `nn.Linear` plus the four-step Attention computation.

The computed weight matrix looks like this:

```
         the   cat   sat
the:    [0.25, 0.28, 0.47]
cat:    [0.28, 0.33, 0.39]
sat:    [0.25, 0.39, 0.36]
```

Each row is one token's attention proportions toward all tokens, summing to 1. Mixing V with these weights yields each token's new vector.

It is called "single-head" because there is only one set of `W_Q, W_K, W_V`, and each token gets only one set of weights. sat's attention to the is 0.25, to cat is 0.39, to itself is 0.36—there is only this one set of numbers, with all attention concerns compressed into it.

### Why Multiple Heads Are Needed

The problem with single-head is: there is only one set of attention weights, so all attention concerns must be compressed into the same set of probabilities. Take "the cat sat on the mat" as an example. When the model processes sat, it must attend to who is doing the action (cat), where the action takes place (mat), and possibly tense (the "sat" before "on" is past tense). These concerns are different in nature, but single-head attention can only give one comprehensive weight allocation—like asking a teacher to grade a dish with only one total score: taste 90, price 60, combined into 75. Information is lost.

If two people answer separately—one specializes in taste, one in value—the information is richer. Multi-head attention does exactly this: allocate multiple sets of weights to each token, so each set can attend to different things.

### How Multi-Head Is Implemented in Code

The most direct idea: since we want 2 heads, define 2 independent sets of `W_Q, W_K, W_V`, and let each compute its own Attention:

```python
# Direct approach: 2 heads, each defines its own set of linear transforms
# head 1
W_Q_1 = nn.Linear(d_model, d_k, bias=False)   # 8 -> 4
W_K_1 = nn.Linear(d_model, d_k, bias=False)
W_V_1 = nn.Linear(d_model, d_k, bias=False)

# head 2
W_Q_2 = nn.Linear(d_model, d_k, bias=False)   # 8 -> 4
W_K_2 = nn.Linear(d_model, d_k, bias=False)
W_V_2 = nn.Linear(d_model, d_k, bias=False)

# Each computes Attention separately
Q_1 = W_Q_1(x)   # [batch, seq_len, 4]
K_1 = W_K_1(x)   # [batch, seq_len, 4]
V_1 = W_V_1(x)   # [batch, seq_len, 4]
# -> head 1 scores -> softmax -> output_1

Q_2 = W_Q_2(x)   # [batch, seq_len, 4]
K_2 = W_K_2(x)   # [batch, seq_len, 4]
V_2 = W_V_2(x)   # [batch, seq_len, 4]
# -> head 2 scores -> softmax -> output_2

# Concatenate
output = cat(output_1, output_2)  # [batch, seq_len, 8]
```

Here `d_model = 8`, and each head's `d_k = 4`. The relationship is `d_k = d_model / num_heads`, or equivalently `d_model = num_heads x d_k`. `d_model` is the unified dimension of the whole model, with both input and output being 8-dimensional; `d_k` is the per-head internal dimension, which only exists within a single head's computation. The `d_k`-dimensional outputs of `num_heads` heads are concatenated to restore `d_model`.

This way of writing is logically clear, but has a practical issue: GPT-3 has 96 heads—do we really write 96 sets of `W_Q, W_K, W_V`? In real implementations, PyTorch merges the parameters of multiple sets into one large matrix, computes all heads' projections in one matrix multiplication, and then splits the result using `view + transpose`:

```python
# Actual implementation: one large matrix, compute once, then split
W_Q = nn.Linear(d_model, d_model, bias=False)   # 8 -> 8 (not 8 -> 4)
Q = W_Q(x)                                       # [batch, seq_len, 8]

# Split the 8 dims into 2 heads x 4 dims
num_heads = 2
d_k = d_model // num_heads                       # 8 // 2 = 4
Q = Q.view(batch, seq_len, num_heads, d_k).transpose(1, 2)
# -> [batch, 2, seq_len, 4]
```

`d_k = d_model // num_heads` is the same relationship as above: the 8 dims are split into 2 groups of 4 each. `view + transpose` only reinterprets the contiguous memory as the shape "2 heads, 4 dims each"—it does no computation. What this step does is exactly equivalent to the "direct approach" above: one `nn.Linear(8, 8)` internally has the same number of parameters as two `nn.Linear(8, 4)` combined.

Each head independently runs a complete Attention computation and gets its own weight matrix:

```
head 1 weights (might focus on syntactic relations):
         the   cat   sat
sat:    [0.05, 0.70, 0.25]   <- high cat weight: who is doing the action?

head 2 weights (might focus on semantic relations):
         the   cat   sat
sat:    [0.10, 0.20, 0.70]   <- high self weight: the meaning of the action itself
```

The two sets of weights each mix their own V, yielding two 4-dimensional output vectors that are concatenated to restore 8 dimensions, then passed through one more $W_O$ linear transform for final integration.

Note that "head 1 focuses on syntax, head 2 on semantics" is only an intuitive understanding—what each head actually learns during training is determined by the data, and does not need to be specified manually. GPT-2 has 12 heads, GPT-3 has 96 heads, and each head can independently learn a distinct attention pattern.

### Complete Data Flow

```
d_model = 8, num_heads = 2 -> per-head dimension d_k = 8 / 2 = 4

input X: [batch, seq_len, 8]
        | multiply by W_Q / W_K / W_V (all 8->8 linear transforms)
Q: [batch, seq_len, 8]
        | view + transpose, split into 2 heads
Q: [batch, 2, seq_len, 4]   <- head 1 takes the first 4 dims, head 2 takes the last 4
        | each head computes Attention independently (scores -> mask -> softmax -> weighted sum)
attn: [batch, 2, seq_len, 4]  <- each head outputs 4 dims
        | transpose + reshape, concatenate back
concat: [batch, seq_len, 8]   <- 2 heads of 4 dims concatenated into 8 dims
        | multiply by W_O (8->8 linear transform)
output: [batch, seq_len, 8]   <- final output, same shape as input
```

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """Multi-head Self-Attention, optionally with a causal mask."""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Project Q, K, and V for all heads in one matrix operation
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        """Map [batch, seq_len, d_model] to the same output shape."""
        batch_size, seq_len, _ = x.shape
        # Project and split into [batch, heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        attn_output = weights @ V
        # Concatenate heads and apply the output projection
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)


In [ ]:
# Test MultiHeadAttention and trace every shape transformation
import torch

torch.manual_seed(42)
d_model, num_heads = 8, 2
d_k = d_model // num_heads
mha = MultiHeadAttention(d_model, num_heads)

# One batch, five Tokens, eight dimensions per Token
test_x = torch.randn(1, 5, d_model)
test_mask = torch.tril(torch.ones(5, 5)).unsqueeze(0).unsqueeze(0)

print("=== MultiHeadAttention Shape Trace ===")
print(f"d_model={d_model}, num_heads={num_heads}, d_k={d_k}")
print()

batch_size, seq_len, _ = test_x.shape
print(f"Input x: {list(test_x.shape)} ← [batch={batch_size}, seq_len={seq_len}, d_model={d_model}]")
print()

# 1. Projection and head split
Q = mha.W_Q(test_x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
K = mha.W_K(test_x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
V = mha.W_V(test_x).view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
print(f"Projected Q: {list(Q.shape)} ← [batch, heads={num_heads}, seq_len={seq_len}, d_k={d_k}]")
print(f"Projected K: {list(K.shape)} ← same shape")
print(f"Projected V: {list(V.shape)} ← same shape")
print(f"  view splits d_model={d_model} into {num_heads}×{d_k}; transpose moves heads forward")
print()

# 2. Attention scores
scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)
print(f"Scores Q@K^T: {list(scores.shape)} ← [batch, heads, seq_len, seq_len]")
print(f"  Each head independently computes a {seq_len}×{seq_len} Attention matrix")
print()

# 3. Mask and Softmax
scores_masked = scores.masked_fill(test_mask == 0, float('-inf'))
weights = torch.nn.functional.softmax(scores_masked, dim=-1)
print(f"Attention weights: {list(weights.shape)} ← unchanged shape; every row sums to 1")
print()

# 4. Weighted sum
attn_out = weights @ V
print(f"Weighted sum weights@V: {list(attn_out.shape)} ← [batch, heads, seq_len, d_k]")
print()

# 5. Concatenate and project
concat = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
output = mha.W_O(concat)
print(f"Concatenate to d_model: {list(concat.shape)} ← {num_heads} heads × {d_k} dimensions = {d_model}")
print(f"Output projection W_O:  {list(output.shape)} ← same shape as input")
print()
print(f"Summary: [{batch_size}, {seq_len}, {d_model}] → [{batch_size}, {seq_len}, {d_model}]")
print("The shape is unchanged, but every Token vector now contains contextual information")


## 5. FFN: How to Process Information After Reading Context

Attention solves "which positions to read information from". But after reading the information, the model still has to keep processing.

Take `the cat sat on the mat`. When the model processes `sat`, Attention can let it read `cat` and `mat`: it knows who the action relates to, and where the action takes place. The problem is that "reading" alone is not enough—the model also has to turn these clues into more useful judgments: `sat` is an action, `cat` is more like a subject, `mat` is more like a location, and when generating the next word it should follow this semantic direction.

This step is handled by the **FFN (Feed-Forward Network)**. You can think of FFN as each position's own small processor: Attention puts the contextual clues into the current vector, and FFN then processes this vector into a new vector better suited for the next layer to use.

The structure of a standard FFN is fixed: first expand the dimension, then activate, then compress back.

```
input x: d_model dims
  |
Linear(d_model -> d_ff)   <- expand, giving the model more room to process
  |
ReLU / GELU              <- add non-linearity; otherwise two Linear layers collapse into one
  |
Linear(d_ff -> d_model)   <- compress back to the Block's unified width
  |
output: d_model dims
```

Why expand first? Because the wider the middle dimension, the richer the variations the model can make. It's like solving a problem: first spread out the scratch paper, then after the computation is done, organize the answer back into the original format.

But FFN has one restriction: it does not let tokens see each other. Positions 0, 1, 2 all use the same set of FFN parameters, but they each compute on their own. Information exchange between tokens has already happened inside Attention.

In [ ]:
# === FFN expands each Token independently and does not mix Tokens ===
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
d_model = 4
d_ff = 16
ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model),)

# Trace the dimensions of one Token through the FFN
x_one_token = torch.tensor([[1.0, -0.5, 0.3, 2.0]])
hidden = ffn[0](x_one_token)
activated = ffn[1](hidden)
out = ffn[2](activated)

print("=== One Token through the FFN ===")
print(f"Input:      {tuple(x_one_token.shape)} ← d_model = 4")
print(f"Expansion:  {tuple(hidden.shape)} ← d_ff = 16")
print(f"Activation: {tuple(activated.shape)} ← same shape, changed values")
print(f"Contraction:{tuple(out.shape)} ← back to d_model = 4")
print()

# Verify that the FFN does not mix different Token positions
x = torch.randn(1, 3, d_model)
out_all = ffn(x)
out_token_1_alone = ffn(x[:, 1:2, :])
is_same = torch.allclose(out_all[:, 1:2, :], out_token_1_alone)

print("=== The FFN Does Not Mix Tokens ===")
print(f"Full input:  {tuple(x.shape)}")
print(f"Full output: {tuple(out_all.shape)}")
print(f"Does position 1 match when computed alone? {is_same}")
print()
print("Key observation: every position independently passes through the same d_model-to-d_model network.")


## 6. Transformer Block

A Transformer Block consists of four components:

- **Attention** (Multi-Head Self-Attention): lets tokens see each other, solving the problem of context information flow
- **FFN** (Feed-Forward Network): applies a deeper transformation at each position independently, solving the problem of how to process information after reading context
- **Residual** (residual connections): `output = input + sublayer output`. Raw information has a direct path through, making deep networks easier to train
- **LayerNorm**: pulls each layer's values back to a stable range, preventing numerical instability as layers increase

The assembly order is:

```
x -> Attention -> Add & Norm -> FFN -> Add & Norm
```

In code:

```python
# Attention sublayer
attn_out = self.attention(x, mask)    # mask passed through to MultiHeadAttention
x = self.norm1(x + attn_out)          # residual connection + LayerNorm

# FFN sublayer
ffn_out = self.ffn(x)
x = self.norm2(x + ffn_out)           # residual connection + LayerNorm
```

The mask comes in as a parameter of `TransformerBlock.forward` and is passed unchanged to `self.attention(x, mask)`. Inside `MultiHeadAttention`, step 3 sets positions where mask is 0 to $-\infty$, so after softmax the weights at those positions become 0, and the token can no longer see the future. FFN does not need a mask—it only applies a per-position transformation (`fc1 -> ReLU -> fc2`) and does not involve any information exchange between tokens.

This is the Post-LN variant: do the sublayer first, then the residual addition and LayerNorm. Real LLMs often use Pre-LN or RMSNorm, which we will upgrade to later.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
    """Two-layer FFN that expands by 4× and contracts back."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class TransformerBlock(nn.Module):
    """Decoder Block: Attention and FFN, each with residual connection and LayerNorm."""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Attention + residual + Norm
        x = self.norm2(x + self.ffn(x))              # FFN + residual + Norm
        return x

# Compare output with and without a causal mask
block = TransformerBlock(d_model=8, num_heads=2)
out_no_mask = block(test_x, mask=None)
out_with_mask = block(test_x, test_mask)

print(f"Input: {test_x.shape} → output: {out_no_mask.shape} ← shape is unchanged")
print()
diff = (out_no_mask - out_with_mask).abs().mean().item()
print(f"Mean difference between unmasked and masked output: {diff:.6f}")
print("→ A nonzero difference confirms that the mask changes available context")
print()
print(f"Position 1 ('cat') without mask: {out_no_mask[0, 1].detach().tolist()}")
print(f"Position 1 ('cat') with mask:    {out_with_mask[0, 1].detach().tolist()}")
print("→ Without a mask, 'cat' can read positions 2–4; with a mask it can read only 0–1")
print("→ The vectors differ because they combine different context ranges")


### Assembly Summary

The full flow of a Transformer Block can be condensed into two lines of code and four keywords:

```
x = LayerNorm(x + Attention(x, mask))    <- stabilize values + preserve raw info + read context
x = LayerNorm(x + FFN(x))                <- stabilize values + preserve raw info + process info
```

Breaking down what each keyword does:

**Stabilize values (LayerNorm)**. After many layers are stacked, vector values may grow larger or smaller. LayerNorm pulls each vector's mean back to 0 and standard deviation back to 1 at every step, keeping the input received by the next layer always in a stable range. Without this step, after a few dozen layers the values could spiral out of control.

**Preserve raw info (residual connection, x +)**. Without `x +`, the sublayer's output would completely replace the raw input. With the residual connection, raw information has a direct path forward, and the sublayer only makes incremental modifications on top of the raw information. This is especially important for deep networks: gradients can flow back directly through the residual path without having to pass through the sublayer's complex computation at every step.

**Read context (Attention)**. Lets each token read information from other tokens based on relevance, blending it into a new vector.

**Process info (FFN)**. Once Attention is done, each token has already read the context. FFN applies a non-linear transformation to each position independently (expand -> activate -> compress), processing the read information into a representation better suited for the next layer.

First look at the overall diagram from the original paper. On the right, the Decoder's `Masked Multi-Head Attention -> Add & Norm -> Feed Forward -> Add & Norm` is exactly the graphical version of the two lines of code above.

![Transformer original paper architecture diagram](https://upload.wikimedia.org/wikipedia/commons/4/49/Attention_Is_All_You_Need_-_Encoder-decoder_Architecture.png)

Next, a simplified Block. Self-Attention is at the bottom, Feed Forward at the top, with Add & Normalize sandwiched in the middle. In other words: first let tokens read context, then let each position process information on its own.

![Transformer Encoder Block simplified diagram](https://jalammar.github.io/images/t/Transformer_encoder.png)

When reading the diagram, do not rush to follow every arrow. Just grasp one main line: the input `x` first goes through Attention, while the original `x` goes through the residual bypass; the two paths are added and then passed through LayerNorm. Then the same thing repeats, except the main computation in the middle switches from Attention to FFN.

Three things to remember:

1. **Each Block has the same input and output shape**: `[batch, seq_len, d_model]` in, `[batch, seq_len, d_model]` out. So Blocks can be stacked like building blocks—GPT-2 stacks 12 layers, GPT-3 stacks 96 layers
2. **The mask only takes effect inside Attention**: passed from the Block to Attention, it masks out future positions on the score matrix. FFN does not need a mask, because it operates position-by-position independently
3. **Residual connections + LayerNorm are "infrastructure"**: residuals give gradients a direct path, LayerNorm keeps values from exploding. They are not the main characters, but without them deep networks cannot be trained

## Summary

What we learned in this section:

- Embedding only gives each token an independent vector; tokens do not communicate
- Attention lets each token read information from context, weighted by relevance
- Q is the query (what am I looking for), K is the label (what am I), V is the content (what can I offer)
- Scaled Dot-Product Attention has four steps: compute scores -> scale -> softmax -> weighted sum
- Causal Mask prevents GPT from peeking at the future—sets future position scores to -inf
- Multi-Head is multiple perspectives in parallel, concatenated at the end
- FFN processes each token independently: expand -> activate -> compress, and does not mix tokens
- Transformer Block = Attention + FFN + Residual + LayerNorm

In the next section, we will stack Transformer Blocks to build a complete Mini-GPT.

## Appendix: RNN vs Transformer

This section answers a deeper question: why is Transformer better suited than RNN for long-range dependencies, and where exactly does it win.

We do not rely on slogans—we directly run two PyTorch modules:

1. `torch.nn.RNN`
2. `torch.nn.MultiheadAttention`

Then we backpropagate from the last position and check whether gradients can reach earlier positions.

**The old approach: RNN passes messages step by step**

When RNN reads a sentence, it passes information forward one word at a time.

```
x1 -> RNN -> h1 -> RNN -> h2 -> RNN -> h3 -> ...
```

`h` can be understood as "what I have remembered so far".

**Problem: long-range information is easily lost**

If the sentence is very long, information from the beginning must pass through many steps to reach the end. The further it travels, the more likely it is to be lost.

This is like the telephone game: after the first sentence is passed 100 times, it has usually become distorted.

**Seeing the problem through experiment first**

The code below demonstrates one thing: the longer the sequence, the smaller the gradient at the beginning positions.

A very small gradient means: the model has difficulty learning from distant earlier context.

**Experiment Step 1: Load PyTorch's built-in RNN**

We will not write the RNN formula by hand here. Instead, we use `torch.nn.RNN` directly. Focus on the experimental phenomenon: can the output from the last position propagate learning signals back to early positions?

In [ ]:
# Experiment 1: load PyTorch's built-in RNN

import torch
import torch.nn as nn
torch.manual_seed(42)

seq_len = 80
batch_size = 1
input_dim = 4
hidden_dim = 8

rnn = nn.RNN(
    input_size=input_dim,
    hidden_size=hidden_dim,
    num_layers=1,
    nonlinearity="tanh",
    batch_first=True,
)

x = torch.randn(batch_size, seq_len, input_dim, requires_grad=True)

print("=== torch.nn.RNN ===")
print(rnn)
print(f"Input shape: {tuple(x.shape)} = [batch, seq_len, input_dim]")
print("This step only prepares the model and input. Next, we inspect hidden states and gradients.")


**Experiment Step 2: Run one forward pass**

`nn.RNN` consumes the entire sequence at once.

It outputs the hidden state at every position, plus the final hidden state.

In [ ]:
# Experiment 2: RNN forward pass
outputs, h_last = rnn(x)

print("=== Forward pass ===")
print(f"outputs shape: {tuple(outputs.shape)}")
print(f"h_last shape:  {tuple(h_last.shape)}")
print()
print("outputs[:, t, :] is the hidden state at position t.")
print("h_last is the hidden state passed out by the final step.")


**Experiment Step 3: Backpropagate from the last step**

We focus on one question: how sensitive is the last step's output to each input position?

The approach: turn the last hidden state into a loss, then call `backward()`.

First look at the gradient values at a few positions, then draw conclusions.

In [ ]:
# Experiment 3: print input gradients
loss = outputs[:, -1, :].pow(2).mean()
loss.backward()

# Input-gradient magnitude at each time position: [seq_len]
grad_by_pos = x.grad.norm(dim=-1).squeeze(0)

print("=== Gradients from last output ===")
print(f"loss from last output: {loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    print(f"position {pos:2d} gradient norm: {grad_by_pos[pos].item():.10f}")

first_grad = grad_by_pos[0].item()
middle_grad = grad_by_pos[seq_len // 2].item()
last_grad = grad_by_pos[-1].item()

print()
print("Values from this run:")
print(f"  gradient at first position:  {first_grad:.3e}")
print(f"  gradient at middle position: {middle_grad:.3e}")
print(f"  gradient at last position:   {last_grad:.3e}")
print()
print("Observation: gradients are largest near the final step and usually approach zero farther back.")
print("This shows that the final step sends only a weak learning signal to distant inputs.")


**Experiment Step 4: View the gradient curve**

The plot below shows the gradient magnitude at each position in the input sequence.

Look at the plot first, then read the conclusion: the curve does not smoothly "gradually decrease"; instead, most earlier positions are nearly flat at zero, with only positions near the end visibly rising.

In [ ]:
import matplotlib.pyplot as plt
# Experiment 4: RNN gradient distribution
import torch
positions = torch.arange(seq_len)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), linewidth=2)
plt.scatter(positions.numpy(), grad_by_pos.detach().numpy(), s=10)
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN input gradient by position")
plt.grid(True, alpha=0.3)
plt.show()

near_end = grad_by_pos[-5:].mean().item()
far_start = grad_by_pos[:5].mean().item()
print("Values corresponding to the plot:")
print(f"  mean gradient over first 5 positions: {far_start:.3e}")
print(f"  mean gradient over last 5 positions:  {near_end:.3e}")
print()
print("Observation: in this run, gradients are concentrated near the end of the sequence.")
print("This is a common difficulty when an RNN learns long-range dependencies: distant positions receive little signal.")


**Experiment Step 5: What happens as the sequence gets longer?**

We keep the RNN configuration the same and only change the sequence length.

Run the curves first, then compare the leftmost and rightmost values of each curve. This plot uses a log scale—one tick mark on a log scale usually means an order-of-magnitude difference.

In [ ]:
import matplotlib.pyplot as plt
# Experiment 5: compare gradients at different sequence lengths
import torch
import torch.nn as nn
def run_rnn_with_length(length):
    """Return the input-gradient magnitude at every position for the same RNN configuration."""
    torch.manual_seed(42)
    model = nn.RNN(input_dim, hidden_dim, nonlinearity="tanh", batch_first=True)
    sample = torch.randn(batch_size, length, input_dim, requires_grad=True)
    out, _ = model(sample)
    loss = out[:, -1, :].pow(2).mean()
    loss.backward()
    return sample.grad.norm(dim=-1).squeeze(0).detach()

lengths = [10, 20, 40, 80]
length_results = {}

plt.figure(figsize=(7, 4))
for length in lengths:
    grads = run_rnn_with_length(length)
    length_results[length] = grads
    relative_pos = torch.linspace(0, 1, steps=length)
    plt.plot(relative_pos.numpy(), grads.numpy(), label=f"seq_len={length}")

plt.xlabel("Relative sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN gradients under different sequence lengths")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Values from this run:")
for length in lengths:
    grads = length_results[length]
    first = grads[0].item()
    last = grads[-1].item()
    ratio = last / max(first, 1e-30)
    print(f"  seq_len={length:2d}: first={first:.3e}, last={last:.3e}, last/first≈{ratio:.3e}")

print()
print("Observation: as the sequence grows, the earliest gradient is usually smaller by more orders of magnitude.")
print("Conclusion: RNNs can process sequences, but long-range learning signals are harder to propagate reliably.")
print("Attention addresses this by giving distant tokens a much shorter path to one another.")


**Experiment Step 6: Switch to PyTorch Attention and check gradients**

The RNN problem we just saw: learning signals from the last step must propagate step by step back to the beginning.

What about Attention? This time we directly use `torch.nn.MultiheadAttention`. Same question: backpropagate only from the last position and check whether earlier positions receive gradients.

We use a causal mask here, so the last position can see all earlier positions, but earlier positions cannot see the future.

In [ ]:
# Experiment 6: inspect input gradients with PyTorch MultiheadAttention
import torch
import torch.nn as nn
torch.manual_seed(42)

attn_dim = 8
num_heads = 2
attention = nn.MultiheadAttention(
    embed_dim=attn_dim,
    num_heads=num_heads,
    batch_first=True,
)

attn_x = torch.randn(batch_size, seq_len, attn_dim, requires_grad=True)

# True means this position is invisible. An upper triangle of True values hides the future.
attn_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

attn_out, attn_weights = attention(
    attn_x,
    attn_x,
    attn_x,
    attn_mask=attn_mask,
    need_weights=True,
    average_attn_weights=False,
)

attn_loss = attn_out[:, -1, :].pow(2).mean()
attn_loss.backward()

attn_grad_by_pos = attn_x.grad.norm(dim=-1).squeeze(0).detach()
last_token_weights = attn_weights[0, :, -1, :].mean(dim=0).detach()

print("=== Gradients from last Attention output ===")
print(f"loss from last output: {attn_loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    grad = attn_grad_by_pos[pos].item()
    weight = last_token_weights[pos].item()
    print(f"position {pos:2d}: grad={grad:.10f}, attention_weight={weight:.10f}")

print()
print("Values from this run:")
print(f"  gradient at first position:  {attn_grad_by_pos[0].item():.3e}")
print(f"  gradient at middle position: {attn_grad_by_pos[seq_len // 2].item():.3e}")
print(f"  gradient at last position:   {attn_grad_by_pos[-1].item():.3e}")
print(f"  sum of the last token's attention weights: {last_token_weights.sum().item():.3f}")
print()
print("Observation: earlier positions no longer rely only on step-by-step propagation; the last position can see them directly.")


**Putting RNN and Attention on the same plot**

Look at the two curves first, then read the conclusion.

The blue line is RNN: gradients at distant positions easily drop close to 0.

The orange line is Attention: earlier position gradients are typically still smaller than the last position, but they are not forced to propagate through many steps—they connect directly to the last position through attention weights.

In [ ]:
import matplotlib.pyplot as plt
# Compare gradients propagated from the last position to every input position
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), label="RNN")
plt.plot(positions.numpy(), attn_grad_by_pos.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("Gradient from last position: RNN vs Attention")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Conclusions from the plot:")
print("  Distant RNN gradients are more likely to shrink nearly to zero.")
print("  Distant Attention gradients may also be small, but they have a direct connection to the last position.")
print("  This is one reason Attention is better suited to learning long-range relationships.")


**Where does the last token actually look?**

Attention doesn't just provide gradients; it also produces attention weights showing how much the last position attends to each other position.

These weights are not hand-crafted—they are computed by `nn.MultiheadAttention` during this forward pass.

In [ ]:
import matplotlib.pyplot as plt
# Attention weights from the final token to the whole sequence
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), last_token_weights.numpy(), linewidth=2)
plt.scatter(positions.numpy(), last_token_weights.numpy(), s=10)
plt.xlabel("Key position")
plt.ylabel("Attention weight")
plt.title("Last token attention weights")
plt.grid(True, alpha=0.3)
plt.show()

print("Conclusions from the plot:")
print("  The final token distributes its attention weight across multiple positions.")
print("  Positions with larger weights influence the final position's output more directly.")


**One more useful plot: cumulative gradient distribution**

The curves above tell us how large the gradient is at each position.

But there is another question: if we sum up gradients across all positions, how much gradient mass is concentrated near the end?

Below we plot cumulative gradient mass. A curve that rises later means gradient mass is more concentrated toward the end.

In [ ]:
import matplotlib.pyplot as plt
# Insight plot: cumulative gradient distribution
import torch
rnn_mass = grad_by_pos / grad_by_pos.sum()
attn_mass = attn_grad_by_pos / attn_grad_by_pos.sum()

rnn_cumsum = torch.cumsum(rnn_mass, dim=0)
attn_cumsum = torch.cumsum(attn_mass, dim=0)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), rnn_cumsum.numpy(), label="RNN")
plt.plot(positions.numpy(), attn_cumsum.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Cumulative gradient mass")
plt.title("Where does the gradient mass accumulate?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

rnn_first_half = rnn_mass[:seq_len // 2].sum().item()
attn_first_half = attn_mass[:seq_len // 2].sum().item()

print("Values from the cumulative distributions:")
print(f"  RNN gradient share in first half:       {rnn_first_half:.3%}")
print(f"  Attention gradient share in first half: {attn_first_half:.3%}")
print()
print("Observation: in this run, the RNN's gradient mass is more concentrated near the end.")
print("Attention still favors the end, but the first half receives a more visible share of the gradient.")


**What do these experiments actually show?**

Note that this does not mean every distant token's gradient under Attention will be large. A more accurate statement is:

1. RNN's distant information must pass through many steps, and learning signals tend to grow weaker along the way.
2. Attention lets the last position directly see all earlier positions, shortening the path.
3. Actual gradient magnitudes still depend on parameters, input, and attention weights.

That is why we later study Q/K/V, masks, and multi-head: they determine exactly how "direct seeing" works in practice.

## Exercises

> You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

**Exercise 1: Attention score**

The first step of Attention is computing relevance scores using query and key.

**Hint**: The dot product of two vectors can be computed with `(q * k).sum()`.

In [ ]:
# Exercise 1: fill in the Attention score
import torch

q = torch.tensor([1.0, 2.0, 0.0])
k = torch.tensor([3.0, 1.0, 4.0])

# TODO: replace the content inside the triple quotes with your code
score = """Compute the dot product of q and k here"""

assert not isinstance(score, str), "Replace the placeholder inside the triple quotes first"
assert score.item() == 5.0, score
print("✅ Exercise 1 passed: you remembered that an attention score is based on the QK dot product")


**Exercise 2: Causal mask**

During GPT generation, future tokens cannot be peeked at, so position `i` can only see positions `0...i`.

**Hint**: `torch.tril(torch.ones(seq_len, seq_len))` generates a lower triangular matrix.

In [ ]:
# Exercise 2: fill in the Causal Mask
import torch

seq_len = 5

# TODO: generate a 5x5 lower-triangular causal mask
mask = """Generate the causal mask here"""

expected = torch.tensor([
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
])
assert not isinstance(mask, str), "Replace the placeholder inside the triple quotes first"
assert torch.equal(mask, expected), mask
print("✅ Exercise 2 passed: you understand why GPT can only see the past")


**Exercise 3: FFN parameter count**

The Feed-Forward Network (FFN) in a Transformer consists of two linear layers. Suppose hidden_size = 512 and the FFN's intermediate dimension d_ff = 2048. Compute the total parameter count of the FFN (including biases).

Hint: W1 is 2048x512 + 2048 biases; W2 is 512x2048 + 512 biases.

In [ ]:
# Exercise 3: calculate the FFN parameter count
hidden_size = 512
d_ff = 2048

# TODO: calculate the parameter count of W1, including its bias
w1_params = None  # d_ff * hidden_size + d_ff

# TODO: calculate the parameter count of W2, including its bias
w2_params = None  # hidden_size * d_ff + hidden_size

# TODO: calculate the total FFN parameter count
total_params = None

assert w1_params is not None
assert w2_params is not None
assert total_params is not None

expected_w1 = d_ff * hidden_size + d_ff
expected_w2 = hidden_size * d_ff + hidden_size
expected_total = expected_w1 + expected_w2

assert w1_params == expected_w1, f'W1 should have {expected_w1} parameters'
assert w2_params == expected_w2, f'W2 should have {expected_w2} parameters'
assert total_params == expected_total, f'The total should be {expected_total}'

print(f'W1: {w1_params:,} parameters')
print(f'W2: {w2_params:,} parameters')
print(f'FFN total: {total_params:,} parameters')
print(f'The FFN has about {total_params / (hidden_size**2):.0f} times hidden_size^2 parameters')
print('The FFN usually accounts for about two thirds of a Transformer Block\'s parameters.')
print(chr(10004) + ' Exercise 3 passed')


## References

- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), 2017 — the original Transformer paper; the definitions of Scaled Dot-Product Attention and Multi-Head Attention both come from this paper
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) — a line-by-line implementation and annotation of the original paper; the variance argument for the scaling factor sqrt(d_k) in this notebook references the derivation from this article
- Karpathy, [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) — a video tutorial building GPT from scratch in code